# 02 — Proper Motions

**Plots:** VPD (Gaia vs BP3M) · error ellipses · parallax distribution · PM vs position

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
# bp3m is installed as a package — no path manipulation needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results, build_gaia_cov,
    bp3m_full_cov, gaia_pm_sigma, bp3m_pm_sigma, vpd_limits,
)
from bp3m.pipeline.catalog_utils import cov2_geom_sigma

DATA = Path(OUTPUT_DIR)
_gaia_files = sorted((DATA / FIELD_NAME / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])
bp3m = load_bp3m_results(DATA / FIELD_NAME / 'BP3M_results')
stars = bp3m['stars']

# Full marginalised covariance
C_full = bp3m_full_cov(bp3m)   # (N, 5, 5)

has_gaia_pm = np.isfinite(gaia['pmra'].values) & np.isfinite(gaia['pmdec'].values)
print(f'Total stars: {len(stars)}   with Gaia PMs: {has_gaia_pm.sum()}')

In [ ]:
# ── Side-by-side VPD: Gaia vs BP3M ───────────────────────────────────────────
pmra_g  = gaia['pmra'].values
pmdec_g = gaia['pmdec'].values
pmra_b  = stars['pmra_bp3m'].values
pmdec_b = stars['pmdec_bp3m'].values

xlim, ylim = vpd_limits(pmra_b, pmdec_b)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, pmra, pmdec, title, color in [
    (axes[0], pmra_g[has_gaia_pm], pmdec_g[has_gaia_pm],
     'Gaia DR3', 'steelblue'),
    (axes[1], pmra_b, pmdec_b, 'BP3M (this work)', 'darkorange'),
]:
    ax.scatter(pmra, pmdec, s=3, alpha=0.4, c=color, rasterized=True)
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.3)
    ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.3)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
    ax.set_title(title)

axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')
fig.suptitle(f'{FIELD_NAME} — Vector Point Diagram', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_vpd.png', dpi=150)
plt.show()

In [ ]:
# ── VPD with error ellipses (bright subset) ───────────────────────────────────
# Show ellipses only for the brightest N stars to avoid overcrowding
N_ELLIPSE = 200
bright = np.argsort(gaia['gmag'].values)[:N_ELLIPSE]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

C_gaia = build_gaia_cov(gaia)

for ax, pmra, pmdec, C_pm, title, fc in [
    (axes[0],
     pmra_g[bright], pmdec_g[bright],
     C_gaia[bright, 2:4, 2:4],
     'Gaia DR3 (inflated)', 'steelblue'),
    (axes[1],
     pmra_b[bright], pmdec_b[bright],
     C_full[bright, 2:4, 2:4],
     'BP3M (this work)', 'darkorange'),
]:
    ax.scatter(pmra, pmdec, s=10, c=fc, alpha=0.7, zorder=3)
    for k in range(len(bright)):
        C2 = C_pm[k]
        # Eigenvalue decomposition for ellipse axes
        vals, vecs = np.linalg.eigh(C2)
        angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
        w, h  = 2 * np.sqrt(np.maximum(vals, 0))
        ell = Ellipse((pmra[k], pmdec[k]), width=w, height=h,
                      angle=angle, fc='none', ec=fc, alpha=0.3, lw=0.5)
        ax.add_patch(ell)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel(r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)')
    ax.set_title(title)

axes[0].set_ylabel(r'$\mu_\delta$ (mas yr$^{-1}$)')
fig.suptitle(f'{FIELD_NAME} — VPD with 1σ error ellipses (N={N_ELLIPSE} brightest)',
             fontsize=12)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_vpd_ellipses.png', dpi=150)
plt.show()

In [ ]:
# ── Parallax distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, vals, sig, title, color in [
    (axes[0],
     gaia['parallax'].values[has_gaia_pm],
     np.sqrt(C_gaia[has_gaia_pm, 4, 4]),
     'Gaia parallax', 'steelblue'),
    (axes[1],
     stars['parallax_bp3m'].values,
     stars['sigma_parallax_bp3m'].values,
     'BP3M parallax', 'darkorange'),
]:
    ok = np.isfinite(vals) & np.isfinite(sig)
    ax.errorbar(np.arange(ok.sum()), np.sort(vals[ok]),
                yerr=sig[ok][np.argsort(vals[ok])],
                fmt='none', ecolor=color, alpha=0.3, elinewidth=0.5)
    ax.axhline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Star rank')
    ax.set_ylabel('Parallax (mas)')
    ax.set_title(title)

plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_parallax.png', dpi=150)
plt.show()

In [ ]:
# ── PM as function of sky position ────────────────────────────────────────────
# Look for any spatial trends in BP3M PMs (systematic residuals)
ok = np.isfinite(pmra_b) & np.isfinite(pmdec_b)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pm, label in [
    (axes[0], pmra_b[ok],  r'$\mu_{\alpha^*}$ (mas yr$^{-1}$)'),
    (axes[1], pmdec_b[ok], r'$\mu_\delta$ (mas yr$^{-1}$)'),
]:
    ra_ok = stars['ra'].values[ok] if 'ra' in stars.columns else gaia['ra'].values[ok]
    sc = ax.scatter(ra_ok, pm, c=gaia['gmag'].values[ok],
                    cmap='viridis_r', s=4, alpha=0.5, rasterized=True)
    plt.colorbar(sc, ax=ax, label='Gaia G (mag)')
    ax.set_xlabel('R.A. (deg)')
    ax.set_ylabel(label)

fig.suptitle(f'{FIELD_NAME} — BP3M PMs vs R.A.', fontsize=12)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_02_pm_vs_position.png', dpi=150)
plt.show()